<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/09_causal_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

### References
Causal Self-Attention - [Link](https://www.kaggle.com/code/aisuko/causal-self-attention)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [2]:
import torch
import math

In [38]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    B, seq, d_k = Q.size()
    mask = 1 - torch.triu(torch.ones(seq, seq), diagonal=1)
    scores = (torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)) * mask
    scores[scores == 0] = float('-inf')

    attention = torch.bmm(torch.softmax(scores, dim=-1), V)
    return attention

In [39]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [40]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.5ms)
  ✅ [2/4] Future tokens don't affect past (2.8ms)
  ✅ [3/4] First position only sees itself (1.2ms)
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.1ms total)
  Progress saved. Run status() to see your dashboard.

